# Case study: Kepler's third law, with units

You give GP_ELITE the semi-major axis and the period of the eight planets, in
metres and in seconds, and you tell it those two units, nothing else. It returns
the law T = c·a^1.5 and the units of c, which it deduces on its own: s/m^1.5.
From c follows the product of the constant of gravitation and the mass of the
Sun, G·M☉ = (2π/c)². Fitted on the four inner planets alone, the law gives
G·M☉ within 0.002 % of the value of the International Astronomical Union, and
the periods of the four outer planets within 0.28 %.

This notebook says where the data come from, what the engine returns, what it
does not find, and checks the law on planets kept out of the fit. Every figure
in the text was measured with `benchmarks/case_studies.py kepler`, for the
seeds 0 to 4 (output in `benchmarks/results_0.9/case_studies/kepler.txt`).

Run the cells in order with **Shift + Enter**. The fits take a few seconds each.

*[🇫🇷 Version française](https://colab.research.google.com/github/ariel95500-create/gp-elite/blob/main/examples/case_studies/kepler_units.fr.ipynb)*

## 1. Install

In [ ]:
!pip install -q gp-elite

import gp_elite
print('GP_ELITE', gp_elite.__version__, 'ready.')

## 2. The data and their source

The table is not typed from a textbook. It is the table of mean orbital
elements of Simon and colleagues (1994, *Astronomy and Astrophysics* 282, 663),
as the astronomy library ERFA distributes it (file `plan94.c` of the source
package pyerfa 2.0.1.5 on PyPI). For each planet it gives the semi-major axis
*a*, in astronomical units, and the mean motion, in arcseconds per thousand
years. The period follows: a full turn, 1 296 000 arcseconds, divided by the
arcseconds per year. The Earth row is the centre of mass of the Earth and the
Moon.

In [ ]:
import os
import numpy as np
import pandas as pd

NAME = 'planets_simon1994.csv'
URL = ('https://raw.githubusercontent.com/ariel95500-create/gp-elite/main/'
       'examples/case_studies/data/' + NAME)
path = os.path.join('data', NAME)
df = pd.read_csv(path if os.path.exists(path) else URL, comment='#')

# 1 296 000 arcseconds in a turn, 1000 years in a millennium
df['period_yr'] = 1296000000.0 / df['mean_motion']
df['a3_over_T2'] = df['a_au']**3 / df['period_yr']**2
print(df[['planet', 'a_au', 'period_yr', 'a3_over_T2']].to_string(index=False))

**Read this before the results.** These are not raw observations. Mean
elements come out of a theory of planetary motion fitted to centuries of
observations, and the last column shows that *a* and *T* already follow
Kepler's law: a³/T² is 0.99994 to 0.99997 from Mercury to Mars, 1.0008 for
Jupiter, and 1.0053 to 1.0055 from Saturn to Neptune. This study does not test
whether Kepler was right. It tests what the engine finds in such a table, and
what it deduces from the units.

## 3. With units: the law and its constant

The engine is told that *a* is in metres and *T* in seconds
(`units=['m'], target_units='s'`), and that the law holds a constant whose
units it must find (`unknown_constant=True`). The conversion uses the
astronomical unit, 149 597 870 700 m exactly (IAU 2012), and the Julian year,
365.25 days.

In [ ]:
from gp_elite import GPEliteRegressor

AU = 149597870700.0          # metres in an astronomical unit (IAU 2012)
YEAR = 365.25 * 86400.0      # seconds in a Julian year
GM_IAU = 1.3271244e20        # m^3/s^2, nominal value (IAU 2015)

a = df['a_au'].to_numpy() * AU
T = df['period_yr'].to_numpy() * YEAR

est = GPEliteRegressor(units=['m'], target_units='s', unknown_constant=True,
                       generations=30, speed='fast', random_state=0,
                       parallel=False)
est.fit(a.reshape(-1, 1), T)

c = est.constant_value_
GM = (2 * np.pi / c) ** 2
print('law      :', est.equation_)
print('constant :', c, est.constant_units_string())
print(f'G * M_sun: {GM:.6e} m^3/s^2   ({100 * (GM / GM_IAU - 1):+.3f} % from the IAU value)')

You should read `5.43909e-10 * X0 * sqrt(X0)`, that is c·a^1.5, with c in
s/m^1.5: the engine found the units of the constant from those of the data. The
seeds 0 to 4 all return this formula.

Under units, a number added to c·a^1.5 would be added to a time without being
one, so the engine cannot write it (compare section 5).

From c, G·M☉ = 1.33447e20 m³/s², 0.55 % above the value of the IAU,
1.3271244e20. The gap comes from the table, not from the search: fitted on all
eight rows, the law is pulled by the long periods of Saturn, Uranus and
Neptune, whose a³/T² stands 0.5 % above that of the inner planets (section 2).

## 4. A held-out check: the inner planets predict the outer ones

The same estimator, fitted on Mercury, Venus, the Earth and Mars only. Jupiter,
Saturn, Uranus and Neptune are never seen: their periods are predicted from
their distances.

In [ ]:
inner, outer = slice(0, 4), slice(4, 8)

est_in = GPEliteRegressor(units=['m'], target_units='s', unknown_constant=True,
                          generations=30, speed='fast', random_state=0,
                          parallel=False)
est_in.fit(a[inner].reshape(-1, 1), T[inner])

c_in = est_in.constant_value_
GM_in = (2 * np.pi / c_in) ** 2
print('law, from Mercury to Mars:', est_in.equation_)
print('constant :', c_in, est_in.constant_units_string())
print(f'G * M_sun: {GM_in:.6e} m^3/s^2   ({100 * (GM_in / GM_IAU - 1):+.4f} % from the IAU value)')
print()
pred = est_in.predict(a[outer].reshape(-1, 1))
for name, p, t in zip(df['planet'][outer], pred, T[outer]):
    print(f'{name:8s} predicted {p / YEAR:9.4f} yr   table {t / YEAR:9.4f} yr   '
          f'gap {100 * (p / t - 1):+.3f} %')

The five seeds return the same constant, c = 5.45416e-10 s/m^1.5, and write the
same law a^1.5 in different ways: seed 0, above, as a product of four roots of
*a*; seeds 1 to 3 as `sqrt(X0) * X0`, `sqrt(X0 * X0 * X0)` or `X0 * sqrt(X0)`;
seed 4 as the difference of two terms in a^1.5,
3.42805e-09·a^1.5 − 2.88264e-09·a^1.5. The constant the engine reports is the
coefficient of the law as it predicts, whatever the spelling.

G·M☉ = 1.327103e20 m³/s², 0.0016 % below the IAU value. The periods of the outer
planets come out 0.043 % too long for Jupiter, and 0.27 % to 0.28 % too long for
Saturn, Uranus and Neptune: the gap their a³/T² already shows in the table.

## 5. Without units

The same table in astronomical units and years, without telling the engine
anything about units.

In [ ]:
x_au = df[['a_au']].to_numpy()
P = df['period_yr'].to_numpy()

plain = GPEliteRegressor(generations=30, speed='fast', random_state=0,
                         parallel=False)
plain.fit(x_au, P)
print('law without units:', plain.equation_)
print()
for name, p, t in zip(df['planet'], plain.predict(x_au), P):
    print(f'{name:8s} gap {100 * (p / t - 1):+.3f} %')

Without units, the engine returns `0.00684983 + 0.997207 * X0 * sqrt(X0)`: the
same power, plus 0.0068 years (2.5 days) that nothing in physics supports. The
search minimises the squared error on the periods, which the long periods
dominate: this formula brings Saturn, Uranus and Neptune within 0.01 %, and
costs 2.6 % on the period of Mercury and 0.8 % on Venus. The five seeds return
it. With units this term cannot be written.

Fitted on the four inner planets only, without units, the engine returns
`-7.16378e-06 + 1.00003 * X0 * sqrt(X0)`, and misses the outer periods by the
same 0.04 % and 0.27 % to 0.28 % as with units.

## 6. What the engine does not find

- **Why a³/T² changes from planet to planet.** In this table it varies by up to
  0.56 %. For Jupiter, whose mass is a thousandth of the Sun's (1/1047, in the
  same file), Newton's form of the law, a³/T² proportional to M☉ + m, expects
  about 0.1 %; the table shows 0.08 %. For Saturn, Uranus and Neptune, whose
  masses are far smaller, the 0.5 % is in the table and the engine cannot say
  where it comes from: a law of *a* alone, with one constant, cannot follow it.
- **G and the mass of the Sun apart.** The data hold only their product.
- **The 2π.** The engine returns c; that c = 2π/√(G·M☉) is physics we bring.

## 7. The five seeds

The seeds 0 to 4 return the same constants, to the last printed digit, in every
fit of this notebook; only the spelling of the law fitted on the inner planets
differs (section 4). The figures were measured on the development branch that follows version 0.9.0,
then again on the package of version 1.0.0, with the same results to the last
printed digit (`benchmarks/results_0.9/release_1.0/m3/`).

The full output: `benchmarks/results_0.9/case_studies/kepler.txt`. The protocol,
written before the measurements: `benchmarks/results_0.9/PLAN_PHASE6.md`.